<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../6.%20advanced_features/16.%20testing_flask_applications.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#8592; Chapter 16: Testing</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#128218; Table of Contents</a>
  <a href='18.%20deployment.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 18: Deployment &#8594;</a>
</div>

# Chapter 17: Caching and Performance -- Making Flask Fast

> *"Your blog gets featured on the front page of a news site. Suddenly 50,000 people load your home page per minute. Each request queries the database, renders templates, and computes data. Without caching, your database collapses. Caching stores computed results so you don't repeat expensive work."*

## 🎯 The Big Picture

Every web request that hits your Flask app goes through a chain of expensive operations:

1. Parse the HTTP request
2. Run Python code in the route handler
3. Query the database (potentially multiple times)
4. Render a Jinja2 template
5. Serialize the response

When that response would be **identical** for many users, you're repeating all that work needlessly. Caching breaks this cycle:

```text
Without cache:
  Request 1  ->  DB query  ->  Template render  ->  Response
  Request 2  ->  DB query  ->  Template render  ->  Response
  Request 3  ->  DB query  ->  Template render  ->  Response
  ... 50,000 requests -> database collapses

With cache (5-min TTL):
  Request 1  ->  DB query  ->  Template render  ->  Store result  ->  Response
  Request 2  ->  Cache hit!  ->  Response (1ms)
  Request 3  ->  Cache hit!  ->  Response (1ms)
  ... 49,999 requests -> database barely touched
```

**The trade-off:** speed vs freshness. Cached data may be slightly stale.

> ❓ **When should I *not* cache something?** Avoid caching data that must always be fresh (account balances, real-time prices), per-user private data in a shared cache, or responses containing secrets. Cache only public, slowly-changing data that's expensive to recompute.

## 🧠 Core Concepts: The "Why"

### Caching is a Whiteboard in Your Kitchen

Instead of going to the grocery store (database) every time you need to know what food you have, you write it on the whiteboard (cache) and check there first. If the whiteboard has the answer, great -- you skip the trip. If not (cache miss), you go to the store and update the whiteboard.

The critical question: **when do you erase the whiteboard?** (Cache invalidation -- widely considered the hardest problem in computer science.)

### Where Caching Helps Most

| Scenario | Cache value | Cache duration |
|----------|-------------|----------------|
| Home page (popular posts) | Rendered HTML | 5 minutes |
| User profile page | HTML fragment | 10 minutes |
| Database query results | Python objects | 60 seconds |
| Navigation menu | HTML | 30 minutes |
| API response | JSON | 2 minutes |
### Where Caching Hurts

- **User-specific data** cached without user variation = data leaks
- **Never-invalidated caches** = stale data users never see updates
- **Caching after a write** without invalidating = old data persists

## ⚡ Syntax & First Usage

### Flask-Caching Setup

In [ ]:

# Install: pip install Flask-Caching
# For Redis: pip install Flask-Caching redis

setup_code = """
from flask import Flask
from flask_caching import Cache

app = Flask(__name__)

# Option 1: SimpleCache -- in-memory, single server, no extra services needed
app.config["CACHE_TYPE"] = "SimpleCache"
app.config["CACHE_DEFAULT_TIMEOUT"] = 300   # 5 minutes default

cache = Cache(app)

# Option 2: Redis cache -- distributed, survives restarts, shared across workers
app.config["CACHE_TYPE"] = "RedisCache"
app.config["CACHE_REDIS_URL"] = "redis://localhost:6379/0"
app.config["CACHE_DEFAULT_TIMEOUT"] = 300

cache = Cache(app)

# Option 3: App factory pattern (init_app)
cache = Cache()   # global, without app

def create_app(config_name="default"):
    app = Flask(__name__)
    app.config["CACHE_TYPE"] = "SimpleCache"
    cache.init_app(app)
    return app
"""

print("Flask-Caching setup options:")
print(setup_code)


### `@cache.cached()` -- Route-Level Caching

In [ ]:

route_caching = """
from flask_caching import Cache

cache = Cache(app, config={"CACHE_TYPE": "SimpleCache"})

@app.route("/popular-posts")
@cache.cached(timeout=300)   # cache the entire response for 5 minutes
def popular_posts():
    # This entire function -- including the DB query and template render --
    # is skipped on cache hits. Only runs once every 5 minutes.
    posts = Post.query.order_by(Post.view_count.desc()).limit(10).all()
    return render_template("popular.html", posts=posts)


@app.route("/")
@cache.cached(timeout=60)    # homepage cached for 1 minute
def index():
    latest_posts  = Post.query.order_by(Post.created_at.desc()).limit(5).all()
    popular_posts = Post.query.order_by(Post.view_count.desc()).limit(5).all()
    return render_template("index.html",
        latest_posts=latest_posts,
        popular_posts=popular_posts
    )


# NOTE: Do NOT cache user-specific pages without varying by user!
# @cache.cached()  # WRONG -- all users see the same profile!
# def my_profile():
#     return render_template("profile.html", user=current_user)
"""

print("Route-level caching:")
print(route_caching)
print()
print("How it works:")
print("  1. Request arrives for /popular-posts")
print("  2. Flask checks cache for key 'view//popular-posts'")
print("  3. Cache HIT: return stored bytes immediately (microseconds)")
print("  4. Cache MISS: run function, store result, return response")


### `@cache.memoize()` -- Function-Level Caching

In [ ]:

memoize_code = """
# @cache.memoize() caches based on the function's arguments
# Different arguments = different cache entries

@cache.memoize(timeout=60)
def get_user_posts(user_id, page=1):
    # Called with user_id=1 -> stored as separate entry from user_id=2
    return Post.query.filter_by(user_id=user_id)\
        .order_by(Post.created_at.desc())\
        .paginate(page=page, per_page=10)


@cache.memoize(timeout=300)
def get_category_posts(category_slug):
    category = Category.query.filter_by(slug=category_slug).first_or_404()
    return Post.query.filter_by(category=category)\
        .order_by(Post.created_at.desc())\
        .limit(20).all()


# Usage in routes:
@app.route("/user/<int:user_id>/posts")
def user_posts(user_id):
    posts = get_user_posts(user_id)   # cached per user_id
    return render_template("user_posts.html", posts=posts)


# Invalidating memoized results when data changes:
@app.route("/post/create", methods=["POST"])
@login_required
def create_post():
    post = Post(...)
    db.session.add(post)
    db.session.commit()
    cache.delete_memoized(get_user_posts, current_user.id)   # clear this user's cache
    return redirect(url_for("main.index"))
"""

print("Function-level memoization:")
print(memoize_code)
print()
print("Key difference from @cached:")
print("  @cached:   one cache entry per URL (same for all users)")
print("  @memoize:  one cache entry per unique set of arguments")


### Cache Invalidation

In [ ]:

invalidation_code = """
# Cache invalidation -- the hardest part

# 1. Delete a specific cached route
cache.delete("view//popular-posts")

# 2. Delete a memoized function call by arguments
cache.delete_memoized(get_user_posts, user_id=42)

# 3. Delete ALL cached memoized calls for a function
cache.delete_memoized(get_user_posts)

# 4. Nuclear option: clear the entire cache
cache.clear()

# Pattern: invalidate on write operations
@posts.route("/post/<int:id>/edit", methods=["POST"])
@login_required
def edit_post(id):
    post = Post.query.get_or_404(id)
    post.title = form.title.data
    post.body  = form.body.data
    db.session.commit()

    # Invalidate caches that contain this post
    cache.delete("view//")               # homepage
    cache.delete("view//popular-posts")  # popular posts list
    cache.delete_memoized(get_user_posts, post.author_id)  # author's posts

    return redirect(url_for("posts.view_post", id=id))
"""

print("Cache invalidation strategies:")
print(invalidation_code)
print()
print("The three hard problems in computer science:")
print("  1. Naming things")
print("  2. Cache invalidation")
print("  3. Off-by-one errors")
print()
print("Tip: prefer short TTLs (60-300s) over perfect invalidation.")
print("Stale data for 5 minutes is usually acceptable.")
print("Manual invalidation is error-prone and easy to forget.")


## 🔬 Deep Dive

### Rate Limiting with Flask-Limiter

In [ ]:

rate_limiting_code = """
# Install: pip install Flask-Limiter
from flask_limiter import Limiter
from flask_limiter.util import get_remote_address

limiter = Limiter(
    app,
    key_func=get_remote_address,   # limit per IP address
    default_limits=["200 per day", "50 per hour"]  # applies to all routes
)

# Override for specific routes:
@app.route("/api/send-email")
@limiter.limit("5 per minute")       # strict limit on expensive operations
def send_email():
    send_transactional_email(...)
    return jsonify({"status": "sent"})


@app.route("/auth/login", methods=["POST"])
@limiter.limit("10 per minute")      # prevent brute force attacks
def login():
    ...


@app.route("/api/search")
@limiter.limit("30 per minute")      # allow reasonable usage
def search():
    ...


# Exempting trusted routes:
@app.route("/health")
@limiter.exempt                       # no limit on health checks
def health():
    return "OK"
"""

print("Rate limiting:")
print(rate_limiting_code)
print()
print("When rate limit exceeded: Flask-Limiter returns 429 Too Many Requests")
print()
print("Key functions to rate-limit in any app:")
limits = [
    ("/auth/login",         "10 per minute",  "Prevent brute force"),
    ("/auth/register",      "5 per minute",   "Prevent mass account creation"),
    ("/api/send-email",     "5 per hour",     "Expensive operation"),
    ("/api/send-sms",       "3 per hour",     "Cost control"),
    ("/api/search",         "60 per minute",  "DB-intensive"),
    ("/api/",               "1000 per hour",  "General API protection"),
]
for route, limit, reason in limits:
    print(f"  {route:<25} {limit:<20} {reason}")


### The N+1 Query Problem

In [ ]:

# The most common database performance problem in Flask/SQLAlchemy

n_plus_1_code = """
# BAD: N+1 queries -- this is the most common perf mistake

posts = Post.query.all()          # 1 query to get all posts

for post in posts:
    print(post.author.username)   # 1 EXTRA QUERY PER POST to load the author!
    # If you have 100 posts: 1 + 100 = 101 queries total!
    # Flask-SQLAlchemy lazy-loads relationships by default
"""

good_code = """
# GOOD: eager loading -- 1 query with a JOIN

from sqlalchemy.orm import joinedload

posts = Post.query.options(joinedload(Post.author)).all()
# 1 query: SELECT posts.*, users.* FROM posts JOIN users ON ...

for post in posts:
    print(post.author.username)   # no extra query! already loaded
"""

detect = """
# Detecting N+1 queries in development:
# Install: pip install flask-debugtoolbar
from flask_debugtoolbar import DebugToolbarExtension

app.config["DEBUG_TB_ENABLED"] = True
app.config["SECRET_KEY"] = "dev-key"
toolbar = DebugToolbarExtension(app)

# The toolbar shows SQL queries per request in the browser.
# 100+ queries on a single page = almost certainly an N+1 problem.
"""

print("The N+1 Problem:")
print(n_plus_1_code)
print("The Fix:")
print(good_code)
print("Detection:")
print(detect)


### Database Indexes

In [ ]:

indexes_code = """
# Without an index, every query on a non-primary-key column does a FULL TABLE SCAN
# With 1 million rows: full scan = slow, index scan = fast

# SQLAlchemy model with indexes:
class Post(db.Model):
    id         = db.Column(db.Integer, primary_key=True)   # auto-indexed
    title      = db.Column(db.String(200), nullable=False)
    body       = db.Column(db.Text, nullable=False)
    created_at = db.Column(db.DateTime, default=datetime.utcnow, index=True)
    #                                                              ^^^^^^^^^
    #                                               index on the column itself

    author_id  = db.Column(db.Integer, db.ForeignKey("users.id"), index=True)
    #                                                               ^^^^^^^^^
    #                           index FK columns -- used in JOINs constantly

    slug       = db.Column(db.String(200), unique=True, index=True)
    #                                      unique=True auto-creates unique index


# Composite index (when querying by multiple columns together):
class Post(db.Model):
    ...
    __table_args__ = (
        db.Index("idx_post_author_created", "author_id", "created_at"),
        # Speeds up: Post.query.filter_by(author_id=x).order_by(Post.created_at)
    )
"""

when_to_index = [
    ("Primary keys",        "Auto-indexed"),
    ("Foreign keys",        "Always index -- used in every JOIN"),
    ("Columns in WHERE",    "Index if queried frequently"),
    ("Columns in ORDER BY", "Index if sorted frequently"),
    ("Unique columns",      "unique=True auto-creates an index"),
    ("columns in LIKE",     "Special full-text indexes needed"),
]

print("Database indexing:")
print(indexes_code)
print()
print("When to add an index:")
for col_type, advice in when_to_index:
    print(f"  {col_type:<25} {advice}")
print()
print("Cost of indexes:")
print("  SELECTs become faster (index lookup vs full scan)")
print("  INSERTs/UPDATEs become slightly slower (index must be updated)")
print("  Each index uses disk space")
print("  Rule of thumb: index columns used in WHERE or JOIN, not all columns")


### ⚖️ SimpleCache vs RedisCache

In [ ]:

comparison = """
+-------------------+-----------------------------+-------------------------------------+
| Aspect            | SimpleCache                 | RedisCache                          |
+-------------------+-----------------------------+-------------------------------------+
| Setup             | Zero -- built-in Python     | Requires Redis server               |
| Shared across     | Single process only         | All Gunicorn workers + servers      |
| Survives restart  | No -- lost on app restart   | Yes -- persisted by Redis           |
| Max size          | Configurable, in RAM        | Configurable, in RAM (with AOF/RDB) |
| Performance       | Extremely fast (in-process) | Very fast (sub-millisecond network) |
| Best for          | Development, single server  | Production, multiple workers        |
| Distributed apps  | Not possible                | Yes -- shared across servers        |
+-------------------+-----------------------------+-------------------------------------+
"""

migration_path = """
# config.py -- switch caches based on environment
class DevelopmentConfig:
    CACHE_TYPE = "SimpleCache"      # no Redis needed in dev

class ProductionConfig:
    CACHE_TYPE = "RedisCache"
    CACHE_REDIS_URL = os.environ["REDIS_URL"]   # set by hosting provider
"""

print("SimpleCache vs RedisCache:")
print(comparison)
print()
print("Environment-based configuration:")
print(migration_path)
print()
print("Practical advice:")
print("  Development:  SimpleCache is fine, keep it simple")
print("  Production (1 server, 1 worker): SimpleCache works")
print("  Production (Gunicorn with -w 4): use RedisCache!")
print("  Each Gunicorn worker is a separate process.")
print("  SimpleCache is per-process -- 4 workers = 4 independent caches!")


### ⚖️ `@cached` vs `@memoize`

In [ ]:

when_to_use = """
@cache.cached()    -- route-level, one entry per URL
    Use when: the entire page response is the same for all users
    Example: /popular-posts, /about, /category/python
    Cache key: the URL path

@cache.memoize()   -- function-level, one entry per argument combination
    Use when: a function's result varies by its arguments
    Example: get_user_posts(user_id), get_category(slug)
    Cache key: function name + arguments hash

    ALSO use memoize when the same function is called from multiple routes
    -- you get reuse across routes automatically.
"""

decision_tree = """
Decision tree:
  Does the response depend on the CURRENT USER?
    Yes -> Don't cache at route level (or vary by user)
    No  -> @cache.cached() on the route

  Is this a function called from multiple places with different args?
    Yes -> @cache.memoize() on the function
    No  -> @cache.cached() on the route might be simpler

  Does the function have complex arguments (objects, lists)?
    Yes -> @cache.memoize() requires make_cache_key override
    No  -> @cache.memoize() works out of the box
"""

print("When to use cached vs memoize:")
print(when_to_use)
print(decision_tree)


## 🧪 What If?

### What if you cache user-specific data without varying by user?

In [ ]:

disaster_scenario = """
DISASTER SCENARIO:
-------------------
@app.route("/dashboard")
@login_required
@cache.cached(timeout=300)   # WRONG! No user variation
def dashboard():
    # Shows user's personal stats, posts, notifications
    return render_template("dashboard.html", user=current_user)

What happens:
  User Alice (id=1) visits /dashboard
  -> Flask runs the function, renders Alice's dashboard
  -> Response is cached under key "view//dashboard"

  User Bob (id=2) visits /dashboard (within 5 minutes)
  -> Cache HIT! Returns Alice's dashboard!
  -> Bob sees Alice's personal data, posts, and notifications
  -> CATASTROPHIC privacy breach

FIX: vary the cache key by user
@app.route("/dashboard")
@login_required
@cache.cached(timeout=300, key_prefix=lambda: f"view//dashboard/user/{current_user.id}")
def dashboard():
    return render_template("dashboard.html", user=current_user)

OR: Simply don't cache user-specific pages at route level.
Cache only the expensive sub-components with @memoize.
"""

print(disaster_scenario)
print("Rule: ALWAYS ask 'Is this response the same for every user?'")
print("If NO -> do not use @cache.cached() without user-specific key")


### What if cache is never invalidated? What if cache goes down?

In [ ]:

print("=== Never Invalidating the Cache ===")
stale_problem = """
Scenario: You cache the list of categories for 24 hours.
An admin adds a new category "Machine Learning".
For 24 hours, no user sees it -- they still see the stale list.
The admin thinks the site is broken.

Solution spectrum:
  1. Short TTL (60-300s): accept slightly stale data, no invalidation needed
  2. Manual invalidation: call cache.clear() or cache.delete() after writes
  3. Event-driven: use signals (blinker) to auto-invalidate on model changes
  4. No caching for data that must be real-time: user notifications, stock prices
"""
print(stale_problem)

print()
print("=== Cache Server Goes Down ===")
cache_failure = """
If Redis is down and your code assumes cache always works:
    posts = cache.get("popular_posts")  # returns None (not an error)
    # If you forget to handle None: TypeError: 'NoneType' is not iterable

Best practice: always write cache-tolerant code
    def get_popular_posts():
        posts = cache.get("popular_posts")
        if posts is None:                           # cache miss OR cache down
            posts = Post.query.order_by(...).all()
            try:
                cache.set("popular_posts", posts)   # best-effort cache write
            except Exception:
                pass                                # continue without cache
        return posts

Flask-Caching's @cache.cached() handles this automatically.
It falls back to calling the function if cache is unavailable.
This is another reason to use decorators over manual cache.get/set.
"""
print(cache_failure)


## 🚀 Real-World Project Link

In our **Blog Application**, caching and performance are applied strategically:

```python
# Cached routes (same for all users):
@app.route("/")
@cache.cached(timeout=300)         # 5 minutes -- homepage with latest/popular posts
def index(): ...

@app.route("/category/<slug>")
@cache.cached(timeout=600)         # 10 minutes -- category post list
def category(slug): ...

# Memoized functions (vary by argument):
@cache.memoize(timeout=120)
def get_popular_posts(limit=10): ...   # invalidated when a post's view_count changes

@cache.memoize(timeout=300)
def get_tag_posts(tag_slug): ...       # invalidated when posts are tagged/untagged
```

**Rate limiting:**
```python
@limiter.limit("10 per minute")   # login -- brute force protection
@limiter.limit("5 per minute")    # register -- spam prevention
@limiter.limit("30 per minute")   # comment API -- spam prevention
```

**Database optimisation:**
- All FK columns have indexes
- `joinedload(Post.author)` on list queries -- eliminates N+1
- Composite index on `(author_id, created_at)` for author's post history

> ❓ **What is a decorator?** A decorator is a function that wraps another function to add behaviour before or after it runs. `@app.route('/')` is shorthand for `index = app.route('/')(index)` — it registers your view function with Flask's URL map without any explicit call.

## 📋 Chapter Summary & Cheat Sheet

### Flask-Caching Setup

```python
from flask_caching import Cache
cache = Cache(app, config={"CACHE_TYPE": "SimpleCache"})     # dev
cache = Cache(app, config={"CACHE_TYPE": "RedisCache",
    "CACHE_REDIS_URL": os.environ["REDIS_URL"]})             # prod
```
### Caching Decorators

```python
@app.route("/popular")
@cache.cached(timeout=300)           # 5 min, all users same response
def popular(): ...

@cache.memoize(timeout=60)
def get_user_posts(user_id): ...     # per-argument cache entry
```
### Cache Invalidation

```python
cache.delete("view//popular")               # specific route
cache.delete_memoized(get_user_posts, 42)   # specific function+args
cache.clear()                               # nuclear option
```
### Rate Limiting

```python
from flask_limiter import Limiter
limiter = Limiter(app, key_func=get_remote_address)

@limiter.limit("10 per minute")
def login(): ...
```
### N+1 Fix

```python
# BAD (N+1):
posts = Post.query.all()
for p in posts: print(p.author.username)  # 1 query per post!

# GOOD (1 query):
from sqlalchemy.orm import joinedload
posts = Post.query.options(joinedload(Post.author)).all()
```

### When to Cache

| Cache it | Don't cache it |
|----------|----------------|
| Same for all users | User-specific pages |
| Expensive DB queries | Real-time data |
| Rarely changes | Data that must be fresh |
| Public pages | Authenticated dashboards |

## 💪 Practice Prompts

**1. Add Caching to a Flask App**
Add Flask-Caching to an existing app using `SimpleCache`. Cache your homepage for 2 minutes and your most-viewed posts page for 5 minutes. Verify it works by adding a `print()` statement inside the cached function -- it should print only on cache misses, not on every request. Then add cache invalidation: when a new post is created, delete the homepage and popular-posts cache.

**2. Fix an N+1 Query**
Create a route that displays 20 posts with their author names. First, implement it naively (lazy-loading, which creates N+1 queries). Install `flask-debugtoolbar` to count the queries. Then add `joinedload(Post.author)` and observe the query count drop from 21 to 1. Measure the response time difference.

**3. Implement Rate Limiting**
Add Flask-Limiter to your app. Rate-limit your login route to 5 attempts per minute, your registration route to 3 per hour, and any external API calls to 10 per minute. Write a test that verifies the 429 response is returned after exceeding the limit (use `with app.test_request_context()` to simulate multiple requests).

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../6.%20advanced_features/16.%20testing_flask_applications.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#8592; Chapter 16: Testing</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#128218; Table of Contents</a>
  <a href='18.%20deployment.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 18: Deployment &#8594;</a>
</div>

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../6. advanced_features/16. testing_flask_applications.ipynb' style='font-weight:bold; font-size:1.05em;'>&larr; Previous</a>
  <a href='../TOC.md' style='font-weight:bold; font-size:1.05em; text-align:center;'>Table of Contents</a>
  <a href='18. deployment.ipynb' style='font-weight:bold; font-size:1.05em;'>Next &rarr;</a>
</div>